### Problem 001: Binary Search (LeetCode 704)

### Problem Definition and Constraints
Given an array of integers `nums` which is sorted in ascending order, and an integer `target`, write a function to search `target` in `nums`. If `target` exists, then return its index. Otherwise, return `-1`. You must write an algorithm with $O(\log n)$ runtime complexity.

* Constraints:
  * 1 <= nums.length <= 10^4
  * -10^4 < nums[i], target < 10^4
  * All integers in `nums` are unique.
  * `nums` is sorted in ascending order.

### Examples
* **Example 1:**
  * Input: `nums = [-1, 0, 2, 4, 6, 8], target = 4`
  * Output: `3`
  * Explanation: 4 exists in nums and its index is 3.
* **Example 2:**
  * Input: `nums = [-1, 0, 2, 4, 6, 8], target = 3`
  * Output: `-1`
  * Explanation: 3 does not exist in nums so return -1.

### Brute Force Approach
The baseline strategy is a standard linear search. We iterate through every element in the array from left to right. If we find the target, we return its index. 
* Time Complexity: $O(n)$ — In the worst-case scenario (the target is at the very end or doesn't exist), we must check every single element.
* Space Complexity: $O(1)$ — No extra memory is used.

### Optimized Approach (Binary Search)
Because the array is strictly sorted, we can use a Binary Search to repeatedly halve our search space. We initialize two pointers: `left` at index 0 and `right` at the last index. In a loop, we calculate the `mid` index. If the value at `mid` matches our target, we return it. If the `mid` value is strictly less than our target, the target must be in the right half, so we update `left = mid + 1`. If the `mid` value is greater, we update `right = mid - 1`.
* Time Complexity: $O(\log n)$ — By eliminating half of the remaining elements on every iteration, the search space shrinks logarithmically. 
* Space Complexity: $O(1)$ — We only maintain three integer pointers (`left`, `right`, `mid`), requiring constant memory.

In [ ]:
from typing import List

class Solution:
    
    # --- BRUTE FORCE APPROACH ---
    def search_brute_force(self, nums: List[int], target: int) -> int:
        for i in range(len(nums)):
            if nums[i] == target:
                return i
        return -1


    # --- OPTIMIZED APPROACH ---
    def search(self, nums: List[int], target: int) -> int:
        # Initialize our boundary pointers
        left = 0
        right = len(nums) - 1
        
        # We loop as long as the search space is valid (pointers haven't crossed)
        while left <= right:
            
            # Calculate the middle index. 
            # Note: (left + right) // 2 works in Python because Python integers don't overflow,
            # but left + ((right - left) // 2) is the universally safe way to write this in C++/Java.
            mid = left + ((right - left) // 2)
            
            if nums[mid] == target:
                # We found the exact target
                return mid
                
            elif nums[mid] < target:
                # The middle value is too small. 
                # We discard the left half by moving the left boundary to mid + 1.
                left = mid + 1
                
            else:
                # The middle value is too large.
                # We discard the right half by moving the right boundary to mid - 1.
                right = mid - 1
                
        # If the loop finishes and we haven't returned, the target isn't in the array
        return -1

### Problem 002: Search a 2D Matrix (LeetCode 74)

### Problem Definition and Constraints
You are given an `m x n` 2D integer array (a matrix) and an integer `target`.
* Each row is sorted in non-decreasing order (smallest to largest).
* The first integer of every row is strictly greater than the last integer of the previous row.
Return `True` if the target exists within the matrix, or `False` otherwise.
* Constraints:
  * 1 <= m, n <= 100
  * -10000 <= matrix[i][j], target <= 10000
  * **Requirement:** Must run in $O(\log(m \cdot n))$ time complexity.

### Examples
* **Example 1:**
  * Input: `matrix = [[1, 2, 4, 8], [10, 11, 12, 13], [14, 20, 30, 40]]`, `target = 10`
  * Output: `True`
* **Example 2:**
  * Input: `matrix = [[1, 2, 4, 8], [10, 11, 12, 13], [14, 20, 30, 40]]`, `target = 15`
  * Output: `False`

### Brute Force Approach
The simplest way to solve this is to ignore the sorted properties entirely and use a nested loop to check every single cell in the matrix one by one until you find the target or run out of cells.
* Time Complexity: $O(m \cdot n)$ — In the worst case, you scan every cell in the `m` rows and `n` columns.
* Space Complexity: $O(1)$ — No extra data structures are used.

### Optimized Approach (Double Binary Search)
Because of the strict sorting rules, the matrix essentially behaves like a single, massive sorted array that was chopped up into rows. We can achieve the required $O(\log(m \cdot n))$ time by running binary search **twice**:
1. **Find the correct row:** We set pointers to the top row and bottom row. We calculate a middle row and check if the target falls within its range (between its first and last values). If the target is smaller than the first value, we discard the bottom half of the rows. If it's larger than the last value, we discard the top half. 
2. **Find the correct column:** Once we isolate the single row where the target *must* be, we run a standard 1D binary search (just like the previous problem) left and right across that specific row.
* Time Complexity: $O(\log m + \log n)$ which mathematically simplifies to $O(\log(m \cdot n))$. Finding the row takes $O(\log m)$ and searching the row takes $O(\log n)$.
* Space Complexity: $O(1)$ — We only store a few integer pointers (`top`, `bot`, `left`, `right`).

In [ ]:
from typing import List

class Solution:
    
    # --- BRUTE FORCE APPROACH ---
    def searchMatrix_brute_force(self, matrix: List[List[int]], target: int) -> bool:
        ROWS = len(matrix)
        COLS = len(matrix[0])
        
        # Linearly scan every cell
        for r in range(ROWS):
            for c in range(COLS):
                if matrix[r][c] == target:
                    return True
                    
        return False

    # --- OPTIMIZED APPROACH (Double Binary Search) ---
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        ROWS = len(matrix)
        COLS = len(matrix[0])
        
        # Phase 1: Binary Search to find the correct row
        top = 0
        bot = ROWS - 1
        
        while top <= bot:
            row = top + ((bot - top) // 2)
            
            # Check if target is greater than the largest value in this row
            if target > matrix[row][-1]:
                top = row + 1
            # Check if target is smaller than the smallest value in this row
            elif target < matrix[row][0]:
                bot = row - 1
            else:
                # The target is within the range of this row. 
                # Break the loop to lock in our 'row' variable.
                break
                
        # If the loop finished and the pointers crossed, the target doesn't fit in ANY row.
        if not (top <= bot):
            return False
            
        # Phase 2: Standard Binary Search within the locked-in row
        left = 0
        right = COLS - 1
        
        while left <= right:
            mid = left + ((right - left) // 2)
            
            if target > matrix[row][mid]:
                left = mid + 1
            elif target < matrix[row][mid]:
                right = mid - 1
            else:
                # We found the exact target
                return True
                
        return False

### Problem 003: Koko Eating Bananas (LeetCode 875)

### Problem Definition and Constraints
Koko loves to eat bananas. There are `n` piles of bananas, and the `i`th pile has `piles[i]` bananas. The guards have gone and will come back in `h` hours.
Koko can decide her eating speed, `k` (bananas per hour). 
* Each hour, she chooses one pile and eats `k` bananas from it.
* If the pile has fewer than `k` bananas, she finishes the pile and stops eating for the rest of that hour (she cannot carry over her speed to another pile in the same hour).
Return the **minimum** integer `k` such that she can eat all the bananas within `h` hours.
* Constraints:
  * 1 <= piles.length <= 1,000
  * piles.length <= h <= 1,000,000
  * 1 <= piles[i] <= 1,000,000,000

### Examples
* **Example 1:**
  * Input: `piles = [1, 4, 3, 2]`, `h = 9`
  * Output: `2`
  * Explanation: 
    * If `k = 2`: Pile 1 takes 1 hr. Pile 4 takes 2 hrs. Pile 3 takes 2 hrs (she eats 2, then 1 the next hour). Pile 2 takes 1 hr. Total = 6 hours. 6 <= 9, so this works. 
    * If `k = 1`: Total time would be 10 hours, which is > 9. Minimum valid `k` is 2.
* **Example 2:**
  * Input: `piles = [25, 10, 23, 4]`, `h = 4`
  * Output: `25`
  * Explanation: Since `h = 4` and there are 4 piles, she only has exactly 1 hour per pile. She must eat the largest pile in 1 hour, so her speed must be at least 25.

### The "Aha!" Moment: Searching Answers, not Indices
Until now, you have used Binary Search to find an *index* in an array. This problem introduces a massive level-up: **Binary Searching the Answer Space.**
We don't know the answer `k`, but we know its boundaries:
* **Minimum possible speed:** 1 banana/hr. (She can't eat 0 bananas).
* **Maximum possible speed:** The size of the largest pile `max(piles)`. (Eating faster than the largest pile doesn't save any extra time, because she can still only eat one pile per hour max).

Our "array" is just the range of numbers from `1` to `max(piles)`. Because speeds are strictly increasing (1, 2, 3, 4...), they are naturally sorted! We can binary search this range to find the perfect speed.

### Brute Force Approach
We start at speed `k = 1` and calculate the total hours required to eat all piles. If it takes too long, we try `k = 2`, then `k = 3`, linearly checking every single speed until we find the first one that takes $\le h$ hours.
* Time Complexity: $O(m \cdot n)$ — Where $m$ is the maximum pile size, and $n$ is the number of piles. If the largest pile has 1 billion bananas, we might loop 1 billion times, scanning the whole array every time.
* Space Complexity: $O(1)$ — No extra memory is used.

### Optimized Approach (Binary Search)
We set a `left` pointer to 1 and a `right` pointer to `max(piles)`. We calculate the middle speed `mid`. We simulate how long it takes to eat all piles at `mid` speed.
* If she finishes $\le h$ hours, `mid` is a valid answer! We save it, but we want the *minimum* speed, so we discard the faster half and keep searching the slower half (`right = mid - 1`).
* If she takes $> h$ hours, she is eating too slowly. We discard the slower half and search the faster half (`left = mid + 1`).
* Time Complexity: $O(n \log m)$ — The binary search takes $O(\log m)$ steps. At each step, we iterate through the $n$ piles to calculate the hours.
* Space Complexity: $O(1)$ — Constant extra memory.

In [ ]:
import math
from typing import List

class Solution:
    
    # --- BRUTE FORCE APPROACH ---
    def minEatingSpeed_brute_force(self, piles: List[int], h: int) -> int:
        # Start checking at speed = 1, up to the size of the largest pile
        for k in range(1, max(piles) + 1):
            total_hours = 0
            for pile in piles:
                # math.ceil() rounds up. E.g., 5 bananas at speed 2 takes 3 hours.
                total_hours += math.ceil(pile / k)
                
            # The first speed that finishes within 'h' hours is naturally the minimum
            if total_hours <= h:
                return k
                
        return -1


    # --- OPTIMIZED APPROACH (Binary Search) ---
    def minEatingSpeed(self, piles: List[int], h: int) -> int:
        # The search space for our speed 'k'
        left = 1
        right = max(piles)
        
        # We will store our best (minimum) valid speed here
        res = right
        
        while left <= right:
            mid_speed = left + ((right - left) // 2)
            
            total_hours = 0
            for pile in piles:
                # Calculate hours needed for this pile at mid_speed
                total_hours += math.ceil(pile / mid_speed)
                
            if total_hours <= h:
                # This speed works! But can we eat slower?
                # Save the current best result, then search the left (slower) half
                res = mid_speed
                right = mid_speed - 1
            else:
                # Too slow! We exceeded the time limit 'h'.
                # We MUST eat faster, so search the right (faster) half
                left = mid_speed + 1
                
        return res

### Problem 004: Find Minimum in Rotated Sorted Array (LeetCode 153)

### Problem Definition and Constraints
You are given an array of integers `nums` of length `n`. The array was originally sorted in strictly increasing order, but it has been rotated anywhere from `1` to `n` times.
* Rotating an array means taking the last element and moving it to the front. 
* For example, rotating `[1, 2, 3, 4, 5]` twice results in `[4, 5, 1, 2, 3]`.
Assume all elements are unique. Return the minimum element in this array.
* Constraints:
  * 1 <= nums.length <= 1000
  * -1000 <= nums[i] <= 1000
  * **Requirement:** Must run in $O(\log n)$ time complexity.

### Examples
* **Example 1:**
  * Input: `nums = [3, 4, 5, 6, 1, 2]`
  * Output: `1`
* **Example 2:**
  * Input: `nums = [4, 5, 0, 1, 2, 3]`
  * Output: `0`
* **Example 3:**
  * Input: `nums = [4, 5, 6, 7]` (Rotated 4 times, which is back to its original state)
  * Output: `4`

### Brute Force Approach
The simplest way is to scan the array from left to right using a basic `for` loop and keep track of the smallest number we see. Or, in Python, just use the built-in `min(nums)` function. 
* Time Complexity: $O(n)$ — We must look at every single number in the worst case.
* Space Complexity: $O(1)$ — No extra memory is used.

### Optimized Approach (Convergent Binary Search)
This problem brings back the exact loop condition conversation we *just* had! 
Because we are NOT looking for a specific target number (like `10`), but rather trying to trap the absolute minimum value, we use **convergent binary search**. The loop condition will strictly be `while left < right:`. 

When you rotate a sorted array, you physically snap it into two separate, perfectly sorted halves. The left half will ALWAYS be entirely greater than the right half. (e.g., in `[4, 5, 6, 1, 2]`, the left half `4, 5, 6` is completely larger than the right half `1, 2`). 

We can compare our `mid` pointer against our `right` pointer to figure out which half we are standing in:
1. If `nums[mid] > nums[right]`: Our `mid` is sitting somewhere in the large left half. The true minimum must be to the right, where the array resets. So, we discard the left half (`left = mid + 1`).
2. If `nums[mid] <= nums[right]`: Our `mid` is sitting in the smaller right half. The `mid` *could* be the minimum itself, or the minimum is further to the left. We discard the right side, but we **keep the mid** (`right = mid`).

When `left` and `right` finally land on the exact same index, they have successfully trapped the minimum value!
* Time Complexity: $O(\log n)$ — We cut the search space in half every iteration.
* Space Complexity: $O(1)$ — Only a few pointers are used.

In [ ]:
from typing import List

class Solution:
    
    # --- BRUTE FORCE APPROACH ---
    def findMin_brute_force(self, nums: List[int]) -> int:
        # In Python, this is O(n) under the hood.
        # Alternatively: iterate through and find the first number smaller than the previous.
        return min(nums)


    # --- OPTIMIZED APPROACH (Convergent Binary Search) ---
    def findMin(self, nums: List[int]) -> int:
        left = 0
        right = len(nums) - 1
        
        # Notice we use `<` and NOT `<=`. 
        # We want the pointers to converge and stop exactly when they land on the same item.
        while left < right:
            mid = left + ((right - left) // 2)
            
            # Compare mid against the rightmost element
            if nums[mid] > nums[right]:
                # The mid value is greater than the right boundary.
                # This means 'mid' is in the larger, left-side sorted portion.
                # The minimum value (the reset point) MUST be to the right.
                left = mid + 1
            else:
                # The mid value is less than or equal to the right boundary.
                # This means 'mid' is in the smaller, right-side sorted portion.
                # The minimum is either at 'mid' or somewhere to the left.
                # We move 'right' to 'mid' to keep 'mid' in our search space.
                right = mid
                
        # The loop breaks exactly when left == right.
        # At this point, both pointers have converged on the minimum value.
        return nums[left]